In [1]:
import numpy as np
from src.data_processing import load_data, fill_missing_values, encode_and_scale, downsample

## Data Preprocessing

In this section, I will preprocess the dataset from the EDA phase. The goal is to convert the data into a fully numerical format that can be used for model training. This involves handling missing values, downsampling imbalanced data, encoding categorical features, and scaling numerical features.

### 1. Load and Downsample Data

**Load the raw training and test data:**
- `aug_train.csv`: Training dataset with target labels
- `aug_test.csv`: Test dataset without target labels

**Address class imbalance:**
- The training dataset has a significant class imbalance (majority class dominates)
- Apply downsampling to balance the classes and prevent model bias toward the majority class

### 2. Handle Missing Values

**Fill missing values using two strategies:**

**Mode imputation** for ordinal/categorical features with meaningful missing patterns:
- `enrolled_university`: Current enrollment status
- `education_level`: Highest education level
- `last_new_job`: Years since last job change
- `experience`: Years of professional experience

**'Unknown' category** for features where missing data is informative:
- `company_size`: Company size (missing → not employed)
- `company_type`: Company type (missing → not employed)
- `major_discipline`: College major (missing → no college degree)
- `gender`: Gender (missing → prefer not to say)

This approach preserves information while ensuring no missing values remain.

### 3. Encode Categorical and Ordinal Features

**Special handling for 'city' (high cardinality):**
- Keep only the **top 15 most frequent cities**
- Group all rare cities (108 cities) into a single category: **'Other_City'**
- Reduces dimensionality from 123 features to 16 features (after one-hot encoding)

**Ordinal encoding** for features with natural ordering:
- `experience`: `<1` → 0, `1` → 1, ..., `>20` → 21
- `company_size`: `<10` → 0, `10/49` → 1, ..., `10000+` → 7
- `last_new_job`: `never` → 0, `1` → 1, ..., `>4` → 5

**Label encoding** for nominal categorical features:
- `gender`, `relevent_experience`, `enrolled_university`, `education_level`, `major_discipline`, `company_type`
- Each unique category is mapped to a unique integer (0, 1, 2, ...)

### 4. Feature Scaling

**Log transformation + Standardization** for `training_hours`:
- `training_hours` has a **right-skewed distribution** with extreme outliers
- Apply **log1p transformation** (log(x+1)) to compress the range and reduce skewness
- Then apply **Z-score standardization** to scale to mean=0, std=1
- This makes the distribution more symmetric and reduces the impact of extreme values

**Standard scaling (Z-score)** for `city_development_index`:
- Already has a symmetric distribution (0.4 - 0.95)
- Standardize to mean=0, std=1 for consistent scaling across features

### 5. Save Processed Data

**Output format:**
- `train_features`: Processed training features (NumPy array)
- `train_target`: Training labels (0 or 1)
- `test_features`: Processed test features (NumPy array)

**Key considerations:**
- All transformations (mappings, scaling parameters) are computed from the **training set only**
- The **same transformations** are applied to the test set to prevent data leakage
- Data is saved in `.npz` format for efficient loading in model training phase

---

**Why these preprocessing steps matter:**
1. **Downsampling** prevents model bias toward the majority class
2. **Missing value handling** preserves information while ensuring completeness
3. **City grouping** reduces dimensionality and prevents overfitting on rare cities
4. **Ordinal encoding** preserves the natural ordering of features (e.g., experience levels)
5. **Log transformation** handles skewed distributions and reduces outlier impact
6. **Standardization** ensures all features have comparable scales for distance-based algorithms

In [2]:
# Load the data
train_data, test_data = load_data('../data/raw/aug_train.csv', '../data/raw/aug_test.csv')

In [3]:
# Downsample the training data
train_data_downsampled = downsample(train_data)

In [4]:
# Fill missing values
train_data_filled, mode_values = fill_missing_values(train_data_downsampled)
test_data_filled, _ = fill_missing_values(test_data, is_train=False, mode_values=mode_values)

In [5]:
# Encode and scale the data
train_data_encoded, mappings, scaling_params = encode_and_scale(train_data_filled)
test_data_encoded, _, _ = encode_and_scale(test_data_filled, is_train=False, mappings=mappings, scaling_params=scaling_params)

In [6]:
# Convert the structured array to a regular NumPy array
train_features = np.array([list(row) for row in train_data_encoded[['city', 'city_development_index', 'gender', 'relevent_experience',
                                                                   'enrolled_university', 'education_level', 'major_discipline',
                                                                   'experience', 'company_size', 'company_type', 'last_new_job',
                                                                   'training_hours']]])
train_target = train_data_encoded['target']

test_features = np.array([list(row) for row in test_data_encoded[['city', 'city_development_index', 'gender', 'relevent_experience',
                                                                 'enrolled_university', 'education_level', 'major_discipline',
                                                                 'experience', 'company_size', 'company_type', 'last_new_job',
                                                                 'training_hours']]])

In [7]:
# Save the processed data
np.savez('../data/processed/processed_data.npz',
         train_features=train_features,
         train_target=train_target,
         test_features=test_features)